In [2]:
import torch
from sympy.stats.rv import probability
from torch import nn
from torch.utils.data import TensorDataset, DataLoader, random_split

4개 feature 데이터를 받아서 이진 분류 작업

PyTorch에서는 Dataset이 sample과 label을 관리하고 DataLoader가 batching/shuffle 등을 담당하는 구조

In [3]:
torch.manual_seed(42)

In [4]:
X = torch.randn(300, 4)

# 숨겨진 실제 규칙
true_logit = ( # 총 1개의 선형식
      1.5 * X[:, 0] # v_1 은 +1.5
    - 2.0 * X[:, 1] # v_2 는 -2.0
    + 0.7 * X[:, 2] # v_3 는 -0.7
    + 0.3 * X[:, 3] # v_4 는 -0.3
    + 0.3 * torch.randn(300) # bias
)

y = (true_logit > 0).float().unsqueeze(1)
y.shape

torch.Size([300, 1])

In [7]:
print("X shape:", X.shape)
print("y shape:", y.shape)
print()
print("X mean:", X.mean(dim=0))
print("X std :", X.std(dim=0))
print()
print("class 1 ratio:", y.mean())

X shape: torch.Size([300, 4])
y shape: torch.Size([300, 1])

X mean: tensor([ 0.0024, -0.0007,  0.0094,  0.0780])
X std : tensor([0.9523, 1.0264, 1.0012, 1.0405])

class 1 ratio: tensor(0.5067)


In [8]:
dataset = TensorDataset(X, y)

print(len(dataset))
print(dataset[0])

300
(tensor([ 1.9269,  1.4873,  0.9007, -2.1055]), tensor([1.]))


In [10]:
train_dataset, val_dataset = random_split(
    dataset,
    [240, 60],
    generator=torch.Generator().manual_seed(42)
)

In [11]:
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size = 32,
    shuffle=False
)

In [12]:
class BinaryClassifier(nn.Module):

    def __init__(self):
        super().__init__()

        # 1번 함수
        self.fc1 = nn.Linear(4, 8)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(8, 1)

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)

        return x

In [15]:
model = BinaryClassifier()

loss_fn = nn.BCEWithLogitsLoss()

optimizer = torch.optim.SGD(
    model.parameters(),
    lr=0.05
)

In [18]:
epochs = 50

for epoch in range(1, epochs + 1):

    model.train()

    train_loss_sum = 0

    for X_batch, y_batch in train_loader:
        # 하나의 batch (32 또는 그 이하) 예측하기
        logits = model(X_batch)

        # 나온 결과에서 손실함수를 적용하여 스칼라 값으로 바꾸기
        loss = loss_fn(logits, y_batch)

        # optimizer에 등록된 파라미터들 초기화해주기
        optimizer.zero_grad()

        # 역전파를 통해 파라미터 grad 모두 구해주기
        loss.backward()

        # 파라미터 학습률만큼 학습시켜주기 (학습률 * 편미분 기울기)
        optimizer.step()

        # 손실 값에 batch 개수만큼 곱해주기
        train_loss_sum += loss.item() * X_batch.size(0)

    # 1에포크 종료 시 학습에 나왔던 손실값들 평균내주기
    train_loss = train_loss_sum / len(train_dataset)

    model.eval()

    val_loss_sum = 0
    correct = 0
    total = 0

    with torch.inference_mode():

        for index, (X_batch, y_batch) in enumerate(val_loader):

            logits = model(X_batch)

            loss = loss_fn(logits, y_batch)

            val_loss_sum += loss.item() * X_batch.size(0)

            # print(f"val {index+1}")
            # print("val_loss_sum:", val_loss_sum)

            probabilities = torch.sigmoid(logits)

            predictions = (probabilities >= 0.5).float()

            correct += (predictions == y_batch).sum().item()
            total += y_batch.numel()

        val_loss = val_loss_sum / len(val_dataset)
        val_accuracy = correct / total

    if epoch == 1 or epoch % 10 == 0:
        print(
            f"epoch={epoch:3d} "
            f"train_loss={train_loss:.4f} "
            f"val_loss={val_loss:.4f} "
            f"val_acc={val_accuracy:.3f}"
        )

epoch=  1 train_loss=0.1103 val_loss=0.0845 val_acc=0.967
epoch= 10 train_loss=0.1062 val_loss=0.0812 val_acc=0.967
epoch= 20 train_loss=0.1027 val_loss=0.0764 val_acc=0.967
epoch= 30 train_loss=0.1000 val_loss=0.0727 val_acc=0.967
epoch= 40 train_loss=0.0979 val_loss=0.0698 val_acc=0.967
epoch= 50 train_loss=0.0954 val_loss=0.0669 val_acc=0.967
